# DCGAN Face Generation with TensorFlow 2.x and Keras

This notebook trains a Deep Convolutional Generative Adversarial Network (DCGAN) on the CelebA face dataset.

Dataset: https://www.kaggle.com/datasets/zuozhaorui/celeba

The implementation includes:
- Kaggle/Colab-friendly dataset download helpers
- `tf.data` preprocessing pipeline for `64 x 64 x 3` images normalized to `[-1, 1]`
- DCGAN Generator and Discriminator based on Radford et al.
- Custom `tf.GradientTape` training loop
- Real-label smoothing
- Per-epoch generated image grids
- GIF creation from generated samples
- Generator and Discriminator loss curves

In [ ]:
# Install imageio if it is missing.
try:
    import imageio.v2 as imageio
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imageio"])
    import imageio.v2 as imageio


In [3]:
# Optional Colab/Kaggle setup instructions:
# 1. Kaggle Notebook: the dataset is usually available under /kaggle/input/celeba.
# 2. Google Colab:
#    - Upload kaggle.json from your Kaggle account settings.
#    - Run the download helper cell below.
#    - Or manually upload/extract the CelebA archive and set DATA_ROOT.

import os
import glob
import zipfile
import shutil
import subprocess
from pathlib import Path
from typing import Iterable, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import imageio.v2 as imageio

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU devices: []


In [4]:
# =========================
# Configuration
# =========================

SEED = 42
LATENT_DIM = 100
IMG_SIZE = 64
CHANNELS = 3
IMAGE_SHAPE = (IMG_SIZE, IMG_SIZE, CHANNELS)

BATCH_SIZE = 128
BUFFER_SIZE = 10_000
EPOCHS = 50

LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL_SMOOTHING = 0.9

NUM_EXAMPLES_TO_GENERATE = 16
OUTPUT_DIR = Path("dcgan_outputs")
SAMPLE_DIR = OUTPUT_DIR / "samples"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
GIF_PATH = OUTPUT_DIR / "dcgan_training_progress.gif"

KAGGLE_DATASET = "zuozhaorui/celeba"

np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# Dataset download / discovery helpers
# =========================

def run_command(command: Iterable[str], check: bool = True) -> subprocess.CompletedProcess:
    """Run a shell command with readable error messages."""
    command = list(command)
    print("Running:", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    return result


def setup_kaggle_credentials(kaggle_json_path: Optional[str] = None) -> None:
    """Configure Kaggle credentials in Colab/local notebooks if kaggle.json is available."""
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    if kaggle_json_path is None:
        candidate_paths = [Path("kaggle.json"), Path("/content/kaggle.json")]
    else:
        candidate_paths = [Path(kaggle_json_path)]

    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        print("Using Kaggle credentials from environment variables.")
        return

    for candidate in candidate_paths:
        if candidate.exists():
            target = kaggle_dir / "kaggle.json"
            shutil.copy(candidate, target)
            target.chmod(0o600)
            print(f"Kaggle credentials configured at {target}")
            return

    print(
        "No kaggle.json found yet. In Colab, upload kaggle.json first, or set "
        "KAGGLE_USERNAME and KAGGLE_KEY environment variables."
    )


def download_celeba_from_kaggle(download_dir: str = "data", kaggle_json_path: Optional[str] = None) -> Path:
    """Download and extract CelebA from Kaggle using the Kaggle API."""
    setup_kaggle_credentials(kaggle_json_path)

    try:
        import kaggle  # noqa: F401
    except ImportError:
        run_command(["python", "-m", "pip", "install", "-q", "kaggle"])

    download_path = Path(download_dir)
    download_path.mkdir(parents=True, exist_ok=True)

    try:
        run_command([
            "kaggle", "datasets", "download",
            "-d", KAGGLE_DATASET,
            "-p", str(download_path),
            "--unzip",
        ])
    except RuntimeError as error:
        raise RuntimeError(
            "CelebA download failed. If you are in Colab, upload your Kaggle API token as "
            "kaggle.json, then rerun this cell. In Kaggle Notebook, add the CelebA dataset "
            "from the right sidebar instead. Original error:\n"
            f"{error}"
        ) from error

    return download_path


def find_image_directory(search_roots: Iterable[str]) -> Path:
    """Find the directory containing the largest number of image files."""
    valid_extensions = {".jpg", ".jpeg", ".png"}
    best_dir = None
    best_count = 0

    for root in search_roots:
        root_path = Path(root)
        if not root_path.exists():
            continue

        for directory, _, filenames in os.walk(root_path):
            count = sum(Path(name).suffix.lower() in valid_extensions for name in filenames)
            if count > best_count:
                best_count = count
                best_dir = Path(directory)

    if best_dir is None or best_count == 0:
        raise FileNotFoundError(
            "Could not find CelebA images. Download/extract the dataset and set DATA_ROOTS correctly."
        )

    print(f"Using image directory: {best_dir}")
    print(f"Found images in directory: {best_count:,}")
    return best_dir


def get_celeba_image_dir(data_roots: Iterable[str], auto_download: bool = True) -> Path:
    """Locate CelebA images; optionally download them from Kaggle if missing."""
    try:
        return find_image_directory(data_roots)
    except FileNotFoundError as error:
        print(error)
        if not auto_download:
            raise
        print("CelebA was not found locally. Attempting Kaggle download now...")
        download_dir = "/content/data" if Path("/content").exists() else "data"
        download_celeba_from_kaggle(download_dir=download_dir)
        return find_image_directory([download_dir, *data_roots])


# Common paths for Kaggle, Colab, and local execution.
# If needed, change DATA_ROOTS manually after download/extraction.
DATA_ROOTS = [
    "/kaggle/input/celeba",
    "/kaggle/input/celeba-dataset",
    "/kaggle/input",
    "/content/data",
    "data",
]

# In Colab, upload kaggle.json before running this cell:
# from google.colab import files
# files.upload()  # choose kaggle.json

IMAGE_DIR = get_celeba_image_dir(DATA_ROOTS, auto_download=True)


In [ ]:
# =========================
# Preprocessing and tf.data pipeline
# =========================

def list_image_files(image_dir: Path) -> list[str]:
    patterns = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    files = []
    for pattern in patterns:
        files.extend(glob.glob(str(image_dir / pattern)))
    files = sorted(files)
    if not files:
        raise ValueError(f"No image files found in {image_dir}")
    return files


def decode_and_preprocess_image(file_path: tf.Tensor) -> tf.Tensor:
    """Load image, resize to 64x64, and normalize pixels to [-1, 1]."""
    image_bytes = tf.io.read_file(file_path)
    image = tf.io.decode_image(image_bytes, channels=CHANNELS, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method=tf.image.ResizeMethod.BILINEAR)
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0
    image.set_shape(IMAGE_SHAPE)
    return image


def build_dataset(image_dir: Path, batch_size: int = BATCH_SIZE) -> tf.data.Dataset:
    file_paths = list_image_files(image_dir)
    print(f"Total images: {len(file_paths):,}")

    dataset = tf.data.Dataset.from_tensor_slices(file_paths)
    dataset = dataset.shuffle(buffer_size=min(BUFFER_SIZE, len(file_paths)), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = build_dataset(IMAGE_DIR, BATCH_SIZE)

sample_batch = next(iter(train_dataset))
print("Batch shape:", sample_batch.shape)
print("Pixel range:", float(tf.reduce_min(sample_batch)), float(tf.reduce_max(sample_batch)))

In [ ]:
# =========================
# Visualize real training images
# =========================

def denormalize_images(images: np.ndarray) -> np.ndarray:
    """Convert images from [-1, 1] float range to [0, 1] for display."""
    return np.clip((images + 1.0) / 2.0, 0.0, 1.0)


def show_image_grid(images: np.ndarray, rows: int = 4, cols: int = 4, title: Optional[str] = None) -> None:
    plt.figure(figsize=(cols * 2, rows * 2))
    if title:
        plt.suptitle(title, fontsize=14)

    images = denormalize_images(images[: rows * cols])
    for index, image in enumerate(images):
        plt.subplot(rows, cols, index + 1)
        plt.imshow(image)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_image_grid(sample_batch.numpy(), rows=4, cols=4, title="Real CelebA Images")

In [ ]:
# =========================
# DCGAN Generator
# =========================

def build_generator(latent_dim: int = LATENT_DIM) -> keras.Model:
    """Build a DCGAN generator that maps z -> 64x64x3 image."""
    model = keras.Sequential(name="generator")

    model.add(layers.Input(shape=(latent_dim,)))
    model.add(layers.Dense(4 * 4 * 1024, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Reshape((4, 4, 1024)))

    model.add(layers.Conv2DTranspose(512, kernel_size=5, strides=2, padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    model.add(layers.Conv2DTranspose(256, kernel_size=5, strides=2, padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    model.add(layers.Conv2DTranspose(128, kernel_size=5, strides=2, padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    model.add(layers.Conv2DTranspose(CHANNELS, kernel_size=5, strides=2, padding="same", activation="tanh"))

    return model


generator = build_generator()
generator.summary()

test_noise = tf.random.normal([1, LATENT_DIM])
test_generated_image = generator(test_noise, training=False)
print("Generated image shape:", test_generated_image.shape)

In [ ]:
# =========================
# DCGAN Discriminator
# =========================

def build_discriminator(input_shape: Tuple[int, int, int] = IMAGE_SHAPE) -> keras.Model:
    """Build a DCGAN discriminator that maps image -> real/fake probability."""
    model = keras.Sequential(name="discriminator")

    model.add(layers.Input(shape=input_shape))

    model.add(layers.Conv2D(128, kernel_size=5, strides=2, padding="same"))
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(256, kernel_size=5, strides=2, padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(512, kernel_size=5, strides=2, padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(1024, kernel_size=5, strides=2, padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation="sigmoid"))

    return model


discriminator = build_discriminator()
discriminator.summary()

print("Discriminator output:", discriminator(test_generated_image, training=False).numpy())

In [ ]:
# =========================
# Losses, optimizers, and checkpoints
# =========================

binary_cross_entropy = keras.losses.BinaryCrossentropy(from_logits=False)

generator_optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=BETA_1)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=BETA_1)


def discriminator_loss(real_output: tf.Tensor, fake_output: tf.Tensor) -> tf.Tensor:
    """BCE discriminator loss with real-label smoothing."""
    real_labels = tf.ones_like(real_output) * REAL_LABEL_SMOOTHING
    fake_labels = tf.zeros_like(fake_output)

    real_loss = binary_cross_entropy(real_labels, real_output)
    fake_loss = binary_cross_entropy(fake_labels, fake_output)
    return real_loss + fake_loss


def generator_loss(fake_output: tf.Tensor) -> tf.Tensor:
    """BCE generator loss; generated images should be classified as real."""
    return binary_cross_entropy(tf.ones_like(fake_output), fake_output)


checkpoint = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
)
checkpoint_manager = tf.train.CheckpointManager(checkpoint, str(CHECKPOINT_DIR), max_to_keep=3)

latest_checkpoint = checkpoint_manager.latest_checkpoint
if latest_checkpoint:
    checkpoint.restore(latest_checkpoint)
    print(f"Restored checkpoint: {latest_checkpoint}")
else:
    print("Training from scratch.")

In [ ]:
# =========================
# Custom training step
# =========================

@tf.function
def train_step(real_images: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    batch_size = tf.shape(real_images)[0]
    noise = tf.random.normal([batch_size, LATENT_DIM])

    with tf.GradientTape() as generator_tape, tf.GradientTape() as discriminator_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    generator_gradients = generator_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradients = discriminator_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
# =========================
# Monitoring utilities
# =========================

fixed_seed = tf.random.normal([NUM_EXAMPLES_TO_GENERATE, LATENT_DIM], seed=SEED)


def save_generated_grid(model: keras.Model, epoch: int, seed: tf.Tensor = fixed_seed, rows: int = 4, cols: int = 4) -> Path:
    """Generate and save a fixed-noise image grid for an epoch."""
    predictions = model(seed, training=False).numpy()
    predictions = denormalize_images(predictions)

    fig = plt.figure(figsize=(cols * 2, rows * 2))
    for index in range(rows * cols):
        plt.subplot(rows, cols, index + 1)
        plt.imshow(predictions[index])
        plt.axis("off")

    plt.tight_layout()
    output_path = SAMPLE_DIR / f"epoch_{epoch:04d}.png"
    plt.savefig(output_path, dpi=120, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    return output_path


def create_training_gif(sample_dir: Path = SAMPLE_DIR, output_path: Path = GIF_PATH, fps: int = 4) -> Path:
    """Compile saved epoch image grids into an animated GIF."""
    image_paths = sorted(sample_dir.glob("epoch_*.png"))
    if not image_paths:
        raise FileNotFoundError(f"No generated sample images found in {sample_dir}")

    frames = [imageio.imread(path) for path in image_paths]
    imageio.mimsave(output_path, frames, fps=fps)
    print(f"GIF saved to: {output_path}")
    return output_path


def plot_losses(generator_losses: list[float], discriminator_losses: list[float]) -> None:
    """Plot Generator and Discriminator loss curves."""
    plt.figure(figsize=(10, 6))
    plt.plot(generator_losses, label="Generator Loss")
    plt.plot(discriminator_losses, label="Discriminator Loss")
    plt.title("DCGAN Training Losses")
    plt.xlabel("Epoch")
    plt.ylabel("Binary Cross-Entropy Loss")
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
# =========================
# Training loop
# =========================

import time


def train(dataset: tf.data.Dataset, epochs: int = EPOCHS) -> Tuple[list[float], list[float]]:
    generator_losses = []
    discriminator_losses = []

    save_generated_grid(generator, epoch=0)

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        epoch_gen_losses = []
        epoch_disc_losses = []

        for step, image_batch in enumerate(dataset, start=1):
            gen_loss, disc_loss = train_step(image_batch)
            epoch_gen_losses.append(float(gen_loss))
            epoch_disc_losses.append(float(disc_loss))

            if step % 100 == 0:
                print(
                    f"Epoch {epoch:03d} | Step {step:04d} | "
                    f"G Loss: {float(gen_loss):.4f} | D Loss: {float(disc_loss):.4f}"
                )

        mean_gen_loss = float(np.mean(epoch_gen_losses))
        mean_disc_loss = float(np.mean(epoch_disc_losses))
        generator_losses.append(mean_gen_loss)
        discriminator_losses.append(mean_disc_loss)

        sample_path = save_generated_grid(generator, epoch=epoch)
        checkpoint_path = checkpoint_manager.save()
        elapsed = time.time() - start_time

        print(
            f"Epoch {epoch:03d}/{epochs} completed in {elapsed:.1f}s | "
            f"G Loss: {mean_gen_loss:.4f} | D Loss: {mean_disc_loss:.4f} | "
            f"Sample: {sample_path} | Checkpoint: {checkpoint_path}"
        )

    return generator_losses, discriminator_losses


# Start training. Increase EPOCHS for better quality faces.
generator_losses, discriminator_losses = train(train_dataset, EPOCHS)

In [ ]:
# =========================
# Final monitoring outputs
# =========================

plot_losses(generator_losses, discriminator_losses)

gif_path = create_training_gif(SAMPLE_DIR, GIF_PATH, fps=4)
print("Training GIF:", gif_path)

# Save final models in Keras format.
generator.save(OUTPUT_DIR / "generator_final.keras")
discriminator.save(OUTPUT_DIR / "discriminator_final.keras")
print(f"Models saved under: {OUTPUT_DIR}")

In [ ]:
# =========================
# Generate fresh faces after training
# =========================

new_noise = tf.random.normal([16, LATENT_DIM])
generated_faces = generator(new_noise, training=False).numpy()
show_image_grid(generated_faces, rows=4, cols=4, title="Generated Faces")